# Spanish tutor: the conversation skill

The tutor holds a conversation in Spanish **inside the vocabulary you know**, and teaches
new words on purpose instead of by accident. This notebook walks through one real
session, using the project's own code (`src/spanish_tutor/`):

| Section | What it covers |
|---|---|
| 1. Choosing words to pre-teach | A topic query against the database, cross-checked against Claude's own suggestions, and the final grounded choice |
| 2. A short conversation | Three learner messages, typed on an English keyboard, including one "¿cómo se dice?" |
| 3. What was logged | The transcript, the word events, two evaluation metrics in SQL, and what the session cost |

```
topic ──> nearest Tatoeba sentences ──> unknown words, ranked ──> Claude picks n ──> taught first

learner message ──> accents restored, words credited (used) ──> readable sentences retrieved
                ──> Claude replies (at most 1 new word) ──> new words taught, all words logged
```

**Before running:** the data pipeline (notebook 01) must have been built, and
`ANTHROPIC_API_KEY` must be in `.env`. A run makes about ten API calls (roughly 10¢), and
Claude's wording differs from run to run.

**Your learning history is untouched:** the session runs on an in-memory copy of your
word bank.

**Licensing note:** the outputs include Tatoeba (CC BY 2.0 FR) and Wiktionary (CC BY-SA)
excerpts and your personal word list. The nbstripout git filter removes outputs from
commits.

In [ ]:
import sqlite3
import warnings

# sentence-transformers imports tqdm, which warns that notebook progress widgets aren't
# installed. Harmless: we don't use progress bars here.
warnings.filterwarnings("ignore", message="IProgress not found")

import pandas as pd

from spanish_tutor import db, lexicon
from spanish_tutor.config import DB_PATH, MODEL
from spanish_tutor.conversation import (
    ClaudeGenerator,
    Ending,
    TranslationTurn,
    Tutor,
    format_lesson,
    reply_to,
    show,
    show_ending,
    show_translation,
)
from spanish_tutor.topics import topic_candidates
from spanish_tutor.vectorstore import open_store
from spanish_tutor.words import LexiconIndex

pd.set_option("display.max_colwidth", 70)

# A throwaway in-memory copy of the word bank: the session below is real, but it never
# touches your learning history.
conn = db.connect(":memory:")
source = sqlite3.connect(DB_PATH)
source.backup(conn)
source.close()

known = db.known_vocabulary(conn)
known_lemmas = {lemma for lemma, _ in known}
print(f"{len(known):,} words you recognize; model: {MODEL}")

In [ ]:
# The pieces a session needs, loaded once (~20 s): the lemmatizer with its Wiktionary
# corrections, the Tatoeba vector store, and Claude.
corrector = lexicon.load_corrector()
store = open_store()
generator = ClaudeGenerator()


def analyze(text):
    return next(lexicon.analyze([text], corrector=corrector))

## 1. Choosing words to pre-teach

A conversation goes better when the learner knows the topic's key words before it
starts. The question is *which* words. Asking a model "what words matter for
gardening?" gets a plausible list, but nothing checks it against real usage or against
what the learner already knows. So the candidates come from the database instead.

### 1.1 The database's candidates

`topics.topic_candidates()`:

1. Retrieves the 2,000 Tatoeba sentences closest in meaning to the topic, from the vector
   store. (Measured 2026-10-05 on 8 topics: the first version used 300, which left most
   topics with only 5-15 candidates and more noise. With 2,000, coincidences fall back
   while real topic words pile up counts: noise in the top 20 fell from 29% to 12%.)
2. Counts the content words (nouns, verbs, adjectives, adverbs) in them that you don't
   know yet.
3. Ranks them by **topic association**, `count × log(count / expected)`. The expected
   count comes from the word's overall frequency, so a word scores high when it appears
   near the topic far more often than chance predicts. Common words that turn up
   everywhere sink; topic words rise. Words seen fewer than 3 times are dropped as
   coincidence.

The example topic is gardening, the topic of the first real session:

In [ ]:
TOPIC = "el jardín"

candidates = topic_candidates(conn, store, TOPIC, known)
frequency = {
    (r["lemma"], r["pos"]): r["frequency_per_million"]
    for r in conn.execute("SELECT lemma, pos, frequency_per_million FROM lexemes")
}
pd.DataFrame(
    [
        (
            rank,
            c.lemma,
            c.pos,
            (c.definition_en or "")[:40],
            c.occurrences,
            round(frequency[c.analysis], 1),
            round(c.score, 1),
        )
        for rank, c in enumerate(candidates, start=1)
    ],
    columns=["rank", "word", "POS", "meaning", "in the 300 sentences", "per million", "score"],
).set_index("rank")

### 1.2 Cross-checking against Claude's own suggestions

For comparison, Claude (a bilingual model) is asked the same question with no candidate
list: which 15 words would an A2 learner most need for this topic? Each suggestion is then
checked against the database: is it a word in the lexicon, do you already know it, and
where did the database rank it?

In [ ]:
free_prompt = (
    f"A Spanish learner at about A2 level is about to have a conversation about: {TOPIC}. "
    "List the 15 Spanish words that would be most useful to learn first for it, as "
    "dictionary forms with a Universal Dependencies part-of-speech tag, written "
    "lemma|POS (for example regar|VERB)."
)
suggested = [w.strip() for w in generator.select_words(free_prompt).words]
suggested_lemmas = {w.partition("|")[0] for w in suggested}
database_rank = {c.lemma: rank for rank, c in enumerate(candidates, start=1)}


def in_lexicon(lemma):
    return conn.execute("SELECT 1 FROM lexemes WHERE lemma = ?", (lemma,)).fetchone() is not None


pd.DataFrame(
    [
        (
            w,
            "yes" if in_lexicon(w.partition("|")[0]) else "no",
            "yes" if w.partition("|")[0] in known_lemmas else "",
            database_rank.get(w.partition("|")[0], ""),
        )
        for w in suggested
    ],
    columns=["Claude suggests", "in the lexicon", "you know it", "database rank"],
)

And the other direction: the database's top candidates that Claude didn't suggest.

In [ ]:
pd.DataFrame(
    [
        (rank, c.key, c.definition_en[:50])
        for rank, c in enumerate(candidates[:15], start=1)
        if c.lemma not in suggested_lemmas
    ],
    columns=["database rank", "word", "meaning"],
)

**Reading the comparison.** The two sources fail in different ways, which is why the tutor
combines them:

- Claude's free list is usually sensible, but it doesn't know your word bank. Words marked
  "you know it" would waste a teaching slot. It also can't say how often a word occurs in
  real sentences.
- The database list is grounded and personal: every candidate is real, unknown to you,
  and attested near the topic. But it includes words that sit near the topic only by
  coincidence of the corpus, such as a common adjective or a word from an idiom.
- Claude can also name relevant words that none of the 2,000 retrieved sentences contained.
  That's the database's blind spot: retrieval only sees what the corpus says near the
  topic.
- Words that **both** lists contain are the strongest candidates.

So the final step lets Claude do what it's good at, judging usefulness, while the
database does what it's good at, grounding. Claude ranks **only the database's
candidates** and can't add words. Any choice not on the list is dropped, and if none of
its choices are on the list, the top scores are used instead. It sees twice as many
candidates as it is asked to choose (at least 25), and it may choose fewer: broad topics
like "el trabajo" have few words that are really about them, and padding the list would
teach noise. When fewer words are taught than requested, the learner is told why.

### 1.3 The final choice, taught before the conversation

In [ ]:
N_WORDS = 5

tutor = Tutor(
    conn,
    generator,
    LexiconIndex(conn, lexicon.load_wiktionary()),
    analyze,
    store=store,
    topic=TOPIC,
)
lessons = tutor.pre_teach(N_WORDS)  # Claude ranks the database's candidates

pd.DataFrame(
    [
        (
            lex.lemma,
            lex.pos,
            database_rank.get(lex.lemma, ""),
            "yes" if lex.lemma in suggested_lemmas else "",
        )
        for lex in tutor.focus
    ],
    columns=["taught today", "POS", "database rank", "also in Claude's free list"],
)

Each word is taught with its definition and an example sentence you can read: every
other word in it is one you know. The example is the stored one if it's readable, or the
closest readable Tatoeba sentence (`vectorstore.find_example`). The other words taught
today count as known, so examples can reinforce each other.

In [ ]:
for item in lessons:
    print(format_lesson(item))

## 2. A short conversation

The words just taught join the vocabulary the tutor is constrained to. The prompt sends
that vocabulary once, frozen for the session, so it stays in the prompt cache; each turn's
note asks the tutor to use the day's words.

Each learner turn:

1. **Typed markers are expanded.** `'a` → á, `~n` → ñ, `:u` → ü, and `?`/`!` directly
   before a word → ¿/¡, so the learner practices correct spelling on an English keyboard.
   Words typed with no accents at all are restored before analysis when exactly one word
   fits (*detras* → *detrás*).
2. **Every word the learner used is credited** to production (`used`, grade 4, or 2 when
   Claude flags it as misused).
3. **3–4 Tatoeba sentences the learner can read**, similar to the message, are added to the
   turn's note as level and topic anchors. This is the RAG step.
4. **Claude replies**, with at most one word outside the vocabulary. If a draft uses two or
   more, it gets one rewrite that names them. Mistakes are recast in the reply and
   explained in a short English note.
5. **Any new word is taught; every other word in the reply is logged as `seen`.**

The tutor opens:

In [ ]:
show(tutor.open())

In [ ]:
def say(message):
    """One learner message, routed as the app routes it (conversation.reply_to): a
    "¿cómo se dice?", a goodbye that ends the conversation, or a conversation turn."""
    print(f"Tú: {message}")
    as_written, turn = reply_to(tutor, message)
    if as_written != message:
        print(f"  (escrito: {as_written})")
    if isinstance(turn, Ending):
        show_ending(turn)
    elif isinstance(turn, TranslationTurn):
        show_translation(turn)
    else:
        show(turn)


say("S'i, tengo un jard'in peque~no detras de mi casa.")

### "¿Cómo se dice ...?"

When the learner doesn't know how to say something, they can ask. This **pauses the
conversation for one turn**. The tutor gives the Spanish and a short explanation, teaches
every new word in it (no one-word limit, because the learner asked), then repeats the
question still waiting for an answer.

The English inside the quotes is never analyzed as the learner's Spanish. In the first
real session it was, and *water* was credited as *wáter*, the Spanish word for toilet.
Both turns are stored with `kind = 'translation'`, so the vocabulary-adherence metric
leaves them out.

**A cost decision behind this turn.** The structured-output schema is part of the cached
prompt, so a separate translation schema needed its own cache entry. Translations
therefore reuse the reply schema: the Spanish goes in the reply and the explanation in
the note. A translation turn reads the conversation's cache instead of re-sending the
~5k-token prefix (measured: about 2.7¢ down to 0.4¢ per request).

In [ ]:
say('?C\'omo se dice "I like to water the lawn"?')

The conversation resumes, and the learner uses the new words:

In [ ]:
say("Me gusta regar el c'esped por la ma~nana, cuando no hace calor.")

### A wrong word, and expressions

Two things the grading has to get right:

- **A wrong word isn't vocabulary.** The learner writes *jugo* (juice) where they mean
  *juego* (I play). Claude flags each mistake as a wrong *form* of the intended word (a
  conjugation or agreement error: credited, with a lower grade) or a *wrong word*. A
  wrong word never enters the production bank, and isn't taught if the learner didn't
  know it.
- **Fixed expressions are one item.** *Sin embargo* and *a veces* are matched as
  expressions (notebook 01 explains how the list was built), so the learner is credited
  with *sin embargo*, never with *embargo* ("seizure").

Section 3.2 shows the resulting events.

In [ ]:
say("Sin embargo, a veces jugo al fútbol con mis amigos.")

### Ending the conversation

A message that ends with a goodbye (*hasta luego*, *adiós*, *nos vemos*…) ends the
conversation; the web app also has a button. The tutor says goodbye, the end time is
stored, and the summary combines numbers computed in SQL from the log
(`sql/queries/session_stats.sql`, including the words used for the first time ever) with
the tutor's notes, one more Claude call, stored in `session_summaries`.

In [ ]:
say("Bueno, tengo que irme. ¡Hasta luego!")

## 3. What was logged

### 3.1 The transcript

Every message is a row in `turns`, with the metrics the evaluation needs: words outside
the vocabulary before and after any rewrite, whether a rewrite happened, tokens and
latency. The learner's text is stored as written after the markers were expanded.

In [ ]:
session_id = tutor.session_id
pd.read_sql_query(
    """
    SELECT turn_no, role, kind, text_es, note_en,
           draft_out_of_bank AS new_draft, final_out_of_bank AS new_final, retried
    FROM turns
    WHERE session_id = ?
    ORDER BY turn_no
    """,
    conn,
    params=(session_id,),
    dtype={"new_draft": "Int64", "new_final": "Int64", "retried": "Int64"},
).set_index("turn_no")

### 3.2 Word events

Each word event links to the turn that caused it (`word_events.turn_id`). The pre-taught
words are logged with the opening turn, under their own source:

In [ ]:
pd.read_sql_query(
    """
    SELECT t.turn_no, t.role, e.source, e.event_type, e.grade,
           GROUP_CONCAT(l.lemma, ', ') AS words
    FROM word_events AS e
    JOIN turns   AS t USING (turn_id)
    JOIN lexemes AS l USING (lexeme_id)
    WHERE t.session_id = ?
    GROUP BY t.turn_no, t.role, e.source, e.event_type, e.grade
    ORDER BY t.turn_no, e.source, e.event_type
    """,
    conn,
    params=(session_id,),
    dtype={"grade": "Int64"},
)

### 3.3 Two evaluation metrics, in plain SQL

Because events link to turns, the Phase 4 metrics are queries, not scripts.

**Vocabulary adherence:** how often the tutor stayed inside the vocabulary (at most one
new word per reply), before and after the rewrite step. Translation turns are excluded.

**Teaching completeness:** every new word in a tutor reply must have a `taught` event
from that same turn. A row in the second result would be a word the learner met but was
never taught.

In [ ]:
adherence = pd.read_sql(
    """
    SELECT COUNT(*)                                          AS tutor_replies,
           SUM(draft_out_of_bank <= 1)                       AS within_limit_first_draft,
           SUM(final_out_of_bank <= 1)                       AS within_limit_final,
           ROUND(AVG(draft_out_of_bank), 2)                  AS avg_new_words_draft,
           SUM(retried)                                      AS rewrites
    FROM turns
    WHERE session_id = ? AND role = 'tutor' AND kind = 'conversation'
    """,
    conn,
    params=(session_id,),
)
gaps = pd.read_sql(
    """
    SELECT t.turn_no, t.final_out_of_bank, COUNT(e.event_id) AS taught
    FROM turns AS t
    LEFT JOIN word_events AS e
           ON e.turn_id = t.turn_id AND e.event_type = 'taught' AND e.source = 'conversation'
    WHERE t.session_id = ? AND t.role = 'tutor' AND t.kind = 'conversation'
    GROUP BY t.turn_id
    HAVING t.final_out_of_bank <> COUNT(e.event_id)
    """,
    conn,
    params=(session_id,),
)
display(adherence)
print("teaching completeness gaps:", len(gaps))

### 3.4 What the session cost

Two cache breakpoints keep input cheap. The instructions and your vocabulary (about 5k
tokens) are cached for 1 hour; the conversation so far is cached for 5 minutes, with the
breakpoint on the newest message. So each turn reads almost everything from the cache, at
5% of the input price, writes only what the last exchange added, and pays full price only
for the per-turn note. (If this notebook ran within the last hour, even the first call
reads the instructions: the prefix is identical.)

Prices are Claude Opus 5.5's at the time of writing: $4 per million input tokens, cache
writes 1.25× (5-minute) or 2× (1-hour), cache reads $0.20, output $20 (thinking
included). The topic-word calls in section 1 and the ending's notes aren't turns, so
they're not shown.

In [ ]:
cost = pd.read_sql(
    """
    SELECT turn_no, kind,
           input_tokens - COALESCE(cache_write_5m_tokens, 0) - COALESCE(cache_write_1h_tokens, 0)
               AS full_price_in,
           cache_write_5m_tokens AS write_5m, cache_write_1h_tokens AS write_1h,
           cache_read_tokens AS cached_in, output_tokens AS out, latency_ms
    FROM turns
    WHERE session_id = ? AND role = 'tutor'
    ORDER BY turn_no
    """,
    conn,
    params=(session_id,),
).set_index("turn_no")
cost["cents"] = (
    (
        cost.full_price_in * 4
        + cost.write_5m * 4 * 1.25
        + cost.write_1h * 4 * 2
        + cost.cached_in * 0.20
        + cost.out * 20
    )
    / 1e6
    * 100
).round(2)
display(cost)
print(f"total: {cost.cents.sum():.1f}¢ for {len(cost)} tutor turns")

## 4. Reading a text (Phase 4)

The reading skill builds on the recommender: open the suggested chapter, study its new
words, read it, then talk about it with the tutor.

1. **Study.** Every word in the text you don't know yet is taught, in batches of 20, in the
   order the words first appear, so each batch covers the next stretch of the text (Jason's
   design). Each batch is a `study` turn carrying its `taught` events (source `reading`).
2. **Read**, at any time. Words not yet studied are underlined on the page; any word can be
   looked up (an unknown one becomes a one-word study turn, a known one is free).
3. **Finished** logs a `seen` event for every known word in the text, and finishes the item,
   which moves a book on to its next chapter.
4. **Talk about it** continues the same session as a conversation. Each turn's note gets the
   three passages of the text closest to what you said (the retrieval step over the text,
   embedded in memory).

This copy of the word bank is first migrated to the latest schema (study turns need
migration 7); your real database is migrated when the app next starts.

In [ ]:
from spanish_tutor import recommend
from spanish_tutor.ingest.expressions import definitions
from spanish_tutor.reading import ReadingSession

db.init_schema(conn)  # the in-memory copy only
suggested = recommend.books(conn)[0]
print(
    f"{suggested['title']}: chapter {suggested['next_chapter_no']}, "
    f"{suggested['next_chapter_title']} ({suggested['next_chapter_new_words']} new words)"
)

reading = ReadingSession(
    conn,
    generator,
    LexiconIndex(conn, lexicon.load_wiktionary(), definitions()),
    analyze,
    suggested["next_content_id"],
    "recommended",
    store=store,
)
pd.DataFrame(
    [(w.lexeme.lemma, w.lexeme.pos, w.sentence, w.context) for w in reading.new_words],
    columns=["new word", "POS", "sentence", "first met in"],
).head(12)

Studying every batch, then finishing. The transcript holds one `study` turn per batch
and a `reading` turn for the finish; the events are logged on them:

In [ ]:
while reading.remaining:
    batch = reading.next_batch()
    reading.study([w.lexeme.lexeme_id for w, _ in batch])
print("sentences fully studied:", reading.readable_until(), "of", len(reading.units))
reading.finish()

pd.read_sql(
    """
    SELECT t.turn_no, t.kind, substr(t.text_es, 1, 60) AS text, e.event_type, COUNT(*) AS events
    FROM turns AS t JOIN word_events AS e ON e.turn_id = t.turn_id
    WHERE t.session_id = ?
    GROUP BY t.turn_id, e.event_type
    ORDER BY t.turn_no
    """,
    conn,
    params=(reading.session_id,),
)

**Talking about it.** The discussion continues the session as a conversation, with the
same vocabulary limit, retries and teaching as any other. For a learner message, the
passages of the text it retrieves look like this:

In [ ]:
talk = reading.discuss(store.embeddings)
message = "El pollo cree que el cielo ha caído en su cabeza."
for passage in talk.passages.search(message):
    print("-", passage[:160])

In [ ]:
show(talk.open())
show(talk.respond(message))

## Summary

| Step | What happens | Code |
|---|---|---|
| Topic candidates | Unknown words near the topic in Tatoeba, ranked by association | `topics.topic_candidates` |
| Selection | Claude ranks the candidates and can't add words; invalid choices are dropped; fewer than requested comes with a reason | `topics.choose_words` |
| Pre-teaching | Definitions and readable examples; the words join the frozen vocabulary | `Tutor.pre_teach`, `teaching.lesson` |
| Typing | Markers expanded; missing accents restored before analysis | `keyboard.expand_markers`, `lexicon.AccentRestorer` |
| Crediting | Words the learner used: `used`, graded; Claude flags wrong forms (credited, lower grade) and wrong words (never credited) | `Tutor.respond`, `Misuse` |
| Expressions | Approved multi-word expressions are one vocabulary item | `lexicon.ExpressionMatcher` |
| Retrieval | Readable Tatoeba sentences near the message, in the turn's note | `vectorstore.search_sentences` |
| Reply | At most one new word, else one rewrite; corrections recast and noted | `Tutor._turn`, `ClaudeGenerator` |
| Translation | "¿Cómo se dice?" pauses for a turn; every new word taught; same cache | `Tutor.translate` |
| Logging | Transcript and metrics in `turns`, including cache writes by lifetime; events linked to turns | `db.add_turn`, `words.log_events` |
| Caching | System prompt cached 1 hour; the conversation 5 minutes, on the newest message | `system_message`, `learner_message` |
| Ending | A goodbye (or the button) ends it: stats from SQL, the tutor's notes stored | `Tutor.end`, `session_stats.sql` |
| Reading | New words studied in batches in order of first appearance; read with lookups; finished logs `seen`; the talk retrieves passages of the text | `reading.ReadingSession`, `reading.TextPassages` |


In [ ]:
conn.close()